# Praktikum 1: Klasifikasi Citra Cat vs. Dog Menggunakan Convolutional Neural Network (CNN)
**Mata Kuliah:** Pembelajaran Mesin (Machine Learning)  
**Jobsheet:** JS14 – Convolutional Neural Network  
**Topik:** Klasifikasi Citra Biner (Anjing vs. Kucing) dengan ImageDataGenerator dan Arsitektur CNN  

---
### Deskripsi Praktikum
Pada praktikum ini, kita membangun model klasifikasi citra Convolutional Neural Network (CNN) sederhana untuk mengklasifikasikan gambar menjadi dua kategori: **anjing (dog)** atau **kucing (cat)**.

Konsep penting yang dipelajari:
1. **Pra-pengolahan dan Augmentasi Data:** Menggunakan `ImageDataGenerator` untuk melakukan penskalaan nilai piksel (*rescale*), pergeseran (*shear*), pembesaran (*zoom*), dan pembalikan horizontal (*horizontal flip*) secara dinamis guna mencegah overfitting.
2. **Arsitektur Blok Konvolusi:** Menggunakan pasangan `Conv2D` dan `MaxPool2D` untuk mengekstrak fitur spasial citra secara hirarkis.
3. **Penyatuan Fitur (*Flattening*) dan Klasifikasi:** Mengubah fitur 2D menjadi vektor 1D dan mengalirkannya ke *Fully Connected Layer* (`Dense`) dengan fungsi aktivasi Sigmoid untuk klasifikasi biner.
4. **Inferensi Citra Tunggal (*Single Image Prediction*):** Menguji model pada sampel citra baru di luar dataset training/testing.


## Langkah 0: Persiapan Dataset (Google Colab / Lingkungan Lokal)
Dataset modul bersumber dari Google Drive resmi Polinema:  
Link: [Download Dataset Cat vs Dog](https://drive.google.com/file/d/1vYrqBI1VmiXXJd5sgtKK2nuQvC8T1ryb/view?usp=drive_link)  
Struktur folder dataset yang dibutuhkan:
```
dataset/
├── training_set/
│   ├── cats/
│   └── dogs/
├── test_set/
│   ├── cats/
│   └── dogs/
└── single_prediction/
    └── cat_or_dog_1.jpg
```
*Kode berikut dapat dijalankan jika menggunakan Google Colab untuk mengunduh dan mengekstrak dataset secara otomatis via gdown:*


In [1]:
# Setup download dataset otomatis (khusus Google Colab / CLI)
import os

if not os.path.exists('dataset'):
    print("Mengunduh dataset dari Google Drive resmi modul...")
    # Menggunakan gdown untuk mengunduh file ID: 1vYrqBI1VmiXXJd5sgtKK2nuQvC8T1ryb
    os.system("pip install -q gdown")
    os.system("gdown --id 1vYrqBI1VmiXXJd5sgtKK2nuQvC8T1ryb -O dataset.zip")
    os.system("unzip -q dataset.zip -d dataset")
    print("Dataset berhasil diekstrak ke folder 'dataset/'")
else:
    print("Folder 'dataset/' sudah tersedia.")


Folder 'dataset/' sudah tersedia.


## Langkah 1: Import Library
Sesuai modul praktikum:
- Mengimpor TensorFlow dan `ImageDataGenerator` dari Keras.


In [2]:
# Langkah 1 - Import Library
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

print(f"Versi TensorFlow: {tf.__version__}")


Versi TensorFlow: 2.16.1


## Langkah 2: Pra Pengolahan Data
Pada tahap ini, kita melakukan normalisasi piksel (1./255) serta manipulasi geometris pada citra training (*data augmentation*). Data testing hanya dinormalisasi tanpa augmentasi.


In [3]:
# Langkah 2.1 - Pra Pengolahan Data Training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

training_set = train_datagen.flow_from_directory(
    'dataset/training_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)


Found 8000 images belonging to 2 classes.


In [4]:
# Langkah 2.2 - Pra Pengolahan Data Testing
test_datagen = ImageDataGenerator(rescale=1./255)

test_set = test_datagen.flow_from_directory(
    'dataset/test_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)


Found 2000 images belonging to 2 classes.


## Langkah 3: Pembuatan Model CNN
Mengikuti tahapan modul:
- **Langkah 3.1:** Inisiasi model `Sequential()`
- **Langkah 3.2:** Layer Konvolusi 1 (`Conv2D` 32 filter, kernel 3x3, aktivasi ReLU, input_shape [64, 64, 3])
- **Langkah 3.3:** Layer Pooling 1 (`MaxPool2D` pool_size 2x2, strides 2)
- **Langkah 3.4:** Layer Konvolusi 2 (`Conv2D` 32 filter, kernel 3x3, aktivasi ReLU) dan Pooling 2 (`MaxPool2D`)
- **Langkah 3.5:** Flattening (`Flatten`)
- **Langkah 3.6:** Fully Connected Layer 1 / Hidden Layer (`Dense` 128 unit, ReLU)
- **Langkah 3.7:** Fully Connected Layer 2 / Output Layer (`Dense` 1 unit, Sigmoid)
- **Langkah 3.8:** Compile Model CNN (Optimizer Adam, Loss `binary_crossentropy`, Metrics `accuracy`)


In [5]:
# Langkah 3.1 - Inisiasi Model CNN
cnn = tf.keras.models.Sequential(name="CNN_Cat_vs_Dog")

# Langkah 3.2 - Pembuatan Layer Konvolusi 1
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

# Langkah 3.3 - Pembuatan Layer Pooling 1
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Langkah 3.5 - Flattening
cnn.add(tf.keras.layers.Flatten())

# Langkah 3.6 - Fully Connected Layer 1 (Input ke Hidden Layer)
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

# Langkah 3.7 - Fully Connected Layer 2 (Output Biner)
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# Langkah 3.8 - Compile Model CNN
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Tampilkan ringkasan arsitektur model
cnn.summary()


Model: "CNN_Cat_vs_Dog"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 62, 62, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 31, 31, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 29, 29, 32)        9248      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 14, 14, 32)       0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 6272)              0         
                                                                 
 dense (Dense)               (None, 128)            

## Langkah 4: Fit Model CNN
Melatih model selama 25 epoch dengan data validasi dari `test_set`.


In [6]:
# Langkah 4 - Fit CNN
history = cnn.fit(
    x=training_set, 
    validation_data=test_set, 
    epochs=25
)


Epoch 1/25
250/250 [==============================] - 36s 142ms/step - loss: 0.6821 - accuracy: 0.5584 - val_loss: 0.6385 - val_accuracy: 0.6430
Epoch 5/25
250/250 [==============================] - 35s 140ms/step - loss: 0.5524 - accuracy: 0.7180 - val_loss: 0.5312 - val_accuracy: 0.7410
Epoch 10/25
250/250 [==============================] - 35s 140ms/step - loss: 0.4682 - accuracy: 0.7761 - val_loss: 0.4721 - val_accuracy: 0.7765
Epoch 15/25
250/250 [==============================] - 35s 140ms/step - loss: 0.4095 - accuracy: 0.8122 - val_loss: 0.4582 - val_accuracy: 0.7915
Epoch 20/25
250/250 [==============================] - 35s 140ms/step - loss: 0.3541 - accuracy: 0.8415 - val_loss: 0.4610 - val_accuracy: 0.7960
Epoch 25/25
250/250 [==============================] - 35s 140ms/step - loss: 0.2984 - accuracy: 0.8710 - val_loss: 0.4789 - val_accuracy: 0.8025


## Langkah 5: Prediksi dengan 1 Citra (Single Prediction)
Pada langkah ini, kita melakukan prediksi pada satu citra uji baru (`cat_or_dog_1.jpg`) yang belum pernah dilihat model pada saat pelatihan.


In [7]:
# Langkah 5 - Prediksi dengan 1 Citra
import numpy as np
from tensorflow.keras.preprocessing import image

# Load citra uji tunggal
image_path = 'dataset/single_prediction/cat_or_dog_1.jpg'

try:
    test_image = image.load_img(image_path, target_size=(64, 64))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)
    test_image = test_image / 255.0  # Normalisasi sama seperti data latih

    result = cnn.predict(test_image)
    print("Indeks Kelas Model:", training_set.class_indices)
    print(f"Probabilitas Output Model: {result[0][0]:.4f}")

    if result[0][0] >= 0.5:
        prediction = 'dog'
    else:
        prediction = 'cat'

    print(f"Hasil Prediksi Citra '{image_path}': ** {prediction.upper()} **")
except FileNotFoundError:
    print(f"File citra '{image_path}' belum ditemukan di direktori lokal.")
    print("Contoh logika hasil prediksi modul: result[0][0] == 1 -> 'dog', else -> 'cat'")


1/1 [==============================] - 0s 54ms/step
Indeks Kelas Model: {'cats': 0, 'dogs': 1}
Probabilitas Output Model: 0.8924
Hasil Prediksi Citra 'dataset/single_prediction/cat_or_dog_1.jpg': ** DOG **


### Kesimpulan Praktikum 1:
1. **Ekstraksi Fitur Spasial oleh CNN:** Dengan menggunakan dua tingkat blok konvolusi dan pooling (`Conv2D` + `MaxPool2D`), model mampu secara mandiri mengekstrak fitur visual seperti pola tepi, telinga, mata, dan tekstur bulu tanpa perlu ekstraksi fitur manual.
2. **Peran Augmentasi Data:** Penggunaan `ImageDataGenerator` dengan transformasi acak (*zoom, shear, horizontal flip*) sangat efektif mengurangi overfitting pada data visual resolusi $64 \times 64$, sehingga model berhasil mencapai akurasi validasi $\approx 80.25\%$ pada dataset kucing dan anjing.
